# 04 - Segmentacao RFM, retencao e recompra

Neste notebook eu separo clientes por recencia, frequencia e valor, e depois olho para recompra e coortes.

In [ ]:
from pathlib import Path
import sqlite3
import urllib.request
import zipfile

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 120)

PROJECT_DIR = Path.cwd()
DB_ZIP_URL = "https://github.com/Urpia-S/Olist_E-commerce_Analytic-SQL-Python/releases/download/data-v1/olist_colab.sqlite.zip"
OUTPUT_DIR = PROJECT_DIR / "outputs_colab"
DB_PATH = PROJECT_DIR / "olist_colab.sqlite"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def baixar_banco_da_release():
    zip_path = PROJECT_DIR / "olist_colab.sqlite.zip"
    urllib.request.urlretrieve(DB_ZIP_URL, zip_path)

    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(PROJECT_DIR)

    print("Banco extraido em:", DB_PATH)


if not DB_PATH.exists():
    baixar_banco_da_release()

conn = sqlite3.connect(DB_PATH)


def consulta(sql):
    return pd.read_sql_query(sql, conn)


def salvar_consulta(sql, arquivo):
    df = consulta(sql)
    destino = OUTPUT_DIR / arquivo
    df.to_csv(destino, index=False)
    print(f"Arquivo salvo: {destino}")
    return df


def grafico_barras(df, x, y, titulo, rotacao=0, top=None):
    dados = df.head(top) if top else df
    ax = dados.plot(kind="bar", x=x, y=y, legend=False, figsize=(10, 4))
    ax.set_title(titulo)
    ax.set_xlabel("")
    ax.set_ylabel(y)
    plt.xticks(rotation=rotacao, ha="right" if rotacao else "center")
    plt.tight_layout()
    plt.show()


consulta("""
-- Objetos disponiveis no banco preparado.
SELECT
    type AS tipo,
    name AS objeto
FROM sqlite_master
WHERE type IN ('table', 'view')
ORDER BY type, name
LIMIT 20;
""")

## 1. Segmentacao RFM

Uso funcoes de janela para transformar recencia, frequencia e valor em scores de 1 a 5.

In [ ]:
rfm_sql = """
-- Primeiro agrego pagamentos por pedido para evitar multiplicar valores no join.
WITH pagamentos AS (
    SELECT order_id, SUM(payment_value) AS valor_pagamento
    FROM core_order_payments
    GROUP BY order_id
),
itens AS (
    SELECT order_id, SUM(price + freight_value) AS valor_itens_com_frete
    FROM core_order_items
    GROUP BY order_id
),
receita_pedido AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp,
        COALESCE(p.valor_pagamento, i.valor_itens_com_frete, 0) AS valor_pedido
    FROM core_orders o
    LEFT JOIN pagamentos p ON p.order_id = o.order_id
    LEFT JOIN itens i ON i.order_id = o.order_id
    WHERE o.order_purchase_timestamp IS NOT NULL
),
-- Consolido as metricas RFM por customer_unique_id.
cliente_metricas AS (
    SELECT
        c.customer_unique_id,
        date(MIN(r.order_purchase_timestamp)) AS data_primeira_compra,
        date(MAX(r.order_purchase_timestamp)) AS data_ultima_compra,
        CAST(
            julianday((SELECT date(MAX(order_purchase_timestamp), '+1 day') FROM core_orders))
            - julianday(date(MAX(r.order_purchase_timestamp)))
            AS INTEGER
        ) AS recencia_dias,
        COUNT(DISTINCT r.order_id) AS frequencia_pedidos,
        ROUND(SUM(r.valor_pedido), 2) AS valor_total
    FROM receita_pedido r
    JOIN core_customers c ON c.customer_id = r.customer_id
    GROUP BY c.customer_unique_id
),
-- NTILE cria scores de 1 a 5 para recencia, frequencia e valor.
scores AS (
    SELECT
        *,
        NTILE(5) OVER (ORDER BY recencia_dias DESC) AS score_recencia,
        NTILE(5) OVER (ORDER BY frequencia_pedidos ASC) AS score_frequencia,
        NTILE(5) OVER (ORDER BY valor_total ASC) AS score_valor,
        RANK() OVER (ORDER BY valor_total DESC) AS rank_valor,
        DENSE_RANK() OVER (ORDER BY frequencia_pedidos DESC) AS rank_frequencia
    FROM cliente_metricas
)
-- A classificacao final transforma os scores em segmentos de cliente.
SELECT
    customer_unique_id,
    data_primeira_compra,
    data_ultima_compra,
    recencia_dias,
    frequencia_pedidos,
    valor_total,
    score_recencia,
    score_frequencia,
    score_valor,
    CAST(score_recencia AS TEXT) || CAST(score_frequencia AS TEXT) || CAST(score_valor AS TEXT) AS score_rfm,
    CASE
        WHEN score_valor >= 4 AND score_frequencia >= 4 THEN 'cliente de alto valor'
        WHEN score_frequencia >= 4 THEN 'cliente recorrente'
        WHEN score_recencia >= 4 THEN 'cliente recente'
        WHEN score_recencia <= 2 AND score_frequencia <= 2 THEN 'cliente inativo'
        ELSE 'cliente ocasional'
    END AS segmento_cliente
FROM scores
ORDER BY score_valor DESC, score_frequencia DESC, score_recencia DESC, valor_total DESC;
"""

segmentacao_rfm_clientes = salvar_consulta(rfm_sql, "segmentacao_rfm_clientes.csv")
segmentacao_rfm_clientes.to_sql("tmp_rfm_clientes", conn, if_exists="replace", index=False)
segmentacao_rfm_clientes.head()

In [ ]:
segmentos = consulta("""
-- Resumo dos segmentos gerados pela RFM.
SELECT
    segmento_cliente,
    COUNT(*) AS clientes,
    ROUND(AVG(valor_total), 2) AS valor_medio,
    ROUND(AVG(frequencia_pedidos), 2) AS frequencia_media
FROM tmp_rfm_clientes
GROUP BY segmento_cliente
ORDER BY clientes DESC;
""")

display(segmentos)
grafico_barras(segmentos, "segmento_cliente", "clientes", "Clientes por segmento RFM", rotacao=30)

## 2. Retencao e recompra

Aqui verifico quantos clientes compram mais de uma vez e acompanho coortes mensais.

In [ ]:
clientes_recorrencia = salvar_consulta("""
-- Separo clientes de compra unica e recorrentes.
SELECT
    CASE WHEN frequencia_pedidos > 1 THEN 'recorrente' ELSE 'compra_unica' END AS tipo_cliente,
    COUNT(*) AS clientes,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS percentual_clientes
FROM (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS frequencia_pedidos
    FROM core_customers c
    JOIN core_orders o ON o.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
) x
GROUP BY CASE WHEN frequencia_pedidos > 1 THEN 'recorrente' ELSE 'compra_unica' END
ORDER BY clientes DESC;
""", "clientes_recorrencia.csv")

coortes_retencao = salvar_consulta("""
-- Coortes: parto do mes de cada compra.
WITH compras AS (
    SELECT
        c.customer_unique_id,
        date(strftime('%Y-%m-01', o.order_purchase_timestamp)) AS mes_compra
    FROM core_orders o
    JOIN core_customers c ON c.customer_id = o.customer_id
    WHERE o.order_purchase_timestamp IS NOT NULL
),
-- Mes da primeira compra define a coorte do cliente.
primeira_compra AS (
    SELECT
        customer_unique_id,
        MIN(mes_compra) AS mes_coorte
    FROM compras
    GROUP BY customer_unique_id
),
coorte_tamanho AS (
    SELECT
        mes_coorte,
        COUNT(*) AS clientes_coorte
    FROM primeira_compra
    GROUP BY mes_coorte
),
-- Atividade mensal depois da primeira compra.
atividade AS (
    SELECT DISTINCT
        p.mes_coorte,
        c.customer_unique_id,
        c.mes_compra,
        (
            (CAST(strftime('%Y', c.mes_compra) AS INTEGER) - CAST(strftime('%Y', p.mes_coorte) AS INTEGER)) * 12
            + (CAST(strftime('%m', c.mes_compra) AS INTEGER) - CAST(strftime('%m', p.mes_coorte) AS INTEGER))
        ) AS meses_desde_primeira_compra
    FROM compras c
    JOIN primeira_compra p ON p.customer_unique_id = c.customer_unique_id
)
SELECT
    strftime('%Y-%m', a.mes_coorte) AS mes_coorte,
    a.meses_desde_primeira_compra,
    ct.clientes_coorte,
    COUNT(DISTINCT a.customer_unique_id) AS clientes_ativos,
    ROUND(100.0 * COUNT(DISTINCT a.customer_unique_id) / NULLIF(ct.clientes_coorte, 0), 2) AS percentual_retencao
FROM atividade a
JOIN coorte_tamanho ct ON ct.mes_coorte = a.mes_coorte
GROUP BY a.mes_coorte, a.meses_desde_primeira_compra, ct.clientes_coorte
ORDER BY a.mes_coorte, a.meses_desde_primeira_compra;
""", "coortes_retencao.csv")

display(clientes_recorrencia)
coortes_retencao.head()

In [ ]:
pivot_retencao = coortes_retencao.pivot(
    index="mes_coorte",
    columns="meses_desde_primeira_compra",
    values="percentual_retencao",
)

plt.figure(figsize=(12, 6))
plt.imshow(pivot_retencao.fillna(0), aspect="auto")
plt.colorbar(label="% retencao")
plt.title("Coortes de retencao por mes da primeira compra")
plt.xlabel("Meses desde a primeira compra")
plt.ylabel("Mes da coorte")
plt.xticks(range(len(pivot_retencao.columns)), pivot_retencao.columns)
plt.yticks(range(len(pivot_retencao.index)), pivot_retencao.index)
plt.tight_layout()
plt.show()